# Order-k — Llama 3.2 3B — All HMMs with KL + R² vs k-suffix

In [ ]:
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import gc
import numba
from tqdm import tqdm
from sklearn.model_selection import train_test_split

sns.set_context('notebook')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.random.seed(42); torch.manual_seed(42)

SEQ_LEN = 20_000
PROBE_START = 15_000
N_SEEDS = 10
TRAIN_FRAC = 0.2
K_VALUES = list(range(1, 21))
CHUNK_SIZE = 4096

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
KL_CSV = os.path.join(RESULTS_DIR, 'kl_llama31_8b.csv')
# R2_CSV not used in KL notebook


## Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'meta-llama/Llama-3.1-8B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    attn_implementation='sdpa', device_map='auto')
model.eval()

N_LAYERS = len(model.model.layers)
LAYERS = list(range(N_LAYERS))
print(f'Model: {MODEL_NAME}')
print(f'Layers: {N_LAYERS}, hidden_size: {model.config.hidden_size}')

for name in ['F', 'Q', 'V']:
    tid = tokenizer.encode(f' {name}', add_special_tokens=False)[-1]
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded="{decoded}"')


## Infrastructure

In [ ]:
def stationary_distribution(T_matrices):
    T_full = sum(T_matrices)
    eigvals, eigvecs = np.linalg.eig(T_full.T)
    idx = np.argmin(np.abs(eigvals - 1.0))
    pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def sample_hmm_sequence(T_matrices, pi, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    n_states, n_tokens = len(pi), len(T_matrices)
    state = rng.choice(n_states, p=pi)
    tokens = []
    for _ in range(seq_len):
        tp = np.array([T_matrices[z][state].sum() for z in range(n_tokens)])
        tp /= tp.sum()
        z = rng.choice(n_tokens, p=tp)
        tokens.append(z)
        nsp = T_matrices[z][state] / T_matrices[z][state].sum()
        state = rng.choice(n_states, p=nsp)
    return np.array(tokens)

def next_token_probs(beliefs, T_matrices):
    n_tokens = len(T_matrices)
    probs = np.zeros((len(beliefs), n_tokens))
    for z in range(n_tokens):
        probs[:, z] = (beliefs @ T_matrices[z]).sum(axis=1)
    return probs

# ===== Tokenization =====

def tokens_to_prompt(tokens, token_names, sep=' '):
    return sep + sep.join(token_names[t] for t in tokens)

def tokenize_prompt(prompt):
    return tokenizer.encode(prompt, return_tensors='pt', truncation=False)

def match_positions(input_ids, tok_ids):
    ids = input_ids[0].cpu().numpy()
    tok_id_set = {tid: zi for zi, tid in enumerate(tok_ids)}
    pos, tok = [], []
    for i, tid in enumerate(ids):
        if tid in tok_id_set:
            pos.append(i); tok.append(tok_id_set[tid])
    return np.array(pos), np.array(tok)

def kl_divergence(p, q, eps=1e-12):
    return np.sum(p * np.log((p + eps) / (q + eps)), axis=-1)

# ===== Numba =====

@numba.njit(cache=True)
def full_bayesian_beliefs_numba(tokens, T_stack, pi):
    n = len(tokens)
    n_states = len(pi)
    beliefs = np.zeros((n, n_states))
    b = pi.copy()
    for t in range(n):
        b = b @ T_stack[tokens[t]]
        s = 0.0
        for j in range(n_states):
            s += b[j]
        if s > 0:
            for j in range(n_states):
                b[j] /= s
        for j in range(n_states):
            beliefs[t, j] = b[j]
    return beliefs

@numba.njit(cache=True)
def reduced_resolution_beliefs_numba(tok_at_pos, start_pos, n, k, table, n_tok):
    n_states = table.shape[1]
    reduced = np.zeros((n, n_states))
    for i in range(n):
        t = start_pos + i
        s = t - k + 1
        if s < 0:
            s = 0
        idx = 0
        for j in range(s, t + 1):
            idx = idx * n_tok + tok_at_pos[j]
        reduced[i] = table[idx]
    return reduced

@numba.njit(cache=True)
def reduced_beliefs_direct_numba(tok_at_pos, start_pos, n, k, T_stack, pi):
    n_states = len(pi)
    reduced = np.zeros((n, n_states))
    for i in range(n):
        t = start_pos + i
        s = t - k + 1
        if s < 0:
            s = 0
        b = pi.copy()
        for j in range(s, t + 1):
            b = b @ T_stack[tok_at_pos[j]]
            sm = 0.0
            for q in range(n_states):
                sm += b[q]
            for q in range(n_states):
                b[q] /= sm
        for q in range(n_states):
            reduced[i, q] = b[q]
    return reduced

def precompute_belief_tables_array(K_VALUES, T_matrices, pi):
    n_tok = len(T_matrices)
    T = np.stack(T_matrices)
    n_states = len(pi)
    max_k = max(K_VALUES)
    prev = pi.reshape(1, n_states)
    tables = {}
    for k in range(1, max_k + 1):
        cur = np.einsum('ps,zsd->pzd', prev, T).reshape(-1, n_states)
        cur /= cur.sum(axis=1, keepdims=True)
        if k in K_VALUES:
            tables[k] = cur
        prev = cur
    return tables

def compute_k_beliefs(tok, probe_start, n, K_VALUES, tables, T_stack, pi, n_tok, max_k_lookup):
    beliefs_by_k = {}
    for k in K_VALUES:
        if k <= max_k_lookup:
            beliefs_by_k[k] = reduced_resolution_beliefs_numba(tok, probe_start, n, k, tables[k], n_tok)
        else:
            beliefs_by_k[k] = reduced_beliefs_direct_numba(tok, probe_start, n, k, T_stack, pi)
    return beliefs_by_k

# ===== Chunked forward =====

def chunked_forward_with_hooks(input_ids, layers, probe_start, collect_hidden=False):
    seq_len = input_ids.shape[1]
    all_hidden = []
    acts = {l: [] for l in layers}
    past_kv = None
    first_hooked_chunk = None

    for start in range(0, seq_len, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, seq_len)
        chunk = input_ids[:, start:end].to(device)
        need_hooks = (end > probe_start)
        hooks = []
        chunk_acts = {}

        if need_hooks:
            if first_hooked_chunk is None:
                first_hooked_chunk = start
            for l in layers:
                def make_hook(li):
                    def fn(module, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        chunk_acts[li] = h.half().cpu()
                    return fn
                hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))

        with torch.no_grad():
            out = model.model(chunk, past_key_values=past_kv, use_cache=True)

        for h in hooks:
            h.remove()
        if need_hooks:
            for l in layers:
                acts[l].append(chunk_acts[l][0])
        if collect_hidden:
            all_hidden.append(out.last_hidden_state[0].cpu())

        past_kv = out.past_key_values
        del out, chunk_acts; torch.cuda.empty_cache()

    del past_kv; torch.cuda.empty_cache()

    if first_hooked_chunk is None:
        first_hooked_chunk = 0
    acts_cat = {l: torch.cat(acts[l], dim=0).numpy() for l in layers}
    hidden_cat = torch.cat(all_hidden, dim=0) if collect_hidden else None

    return acts_cat, hidden_cat, first_hooked_chunk

def extract_late_activations(acts_cat, pos_indices, n_matched, probe_start, first_hooked_chunk, layers):
    late_mask = np.arange(n_matched) >= probe_start
    late_pos_idx = pos_indices[late_mask] - first_hooked_chunk
    n_late = int(late_mask.sum())
    acts_final = {l: acts_cat[l][late_pos_idx] for l in layers}
    return acts_final, n_late

def compute_fullvocab_kl(hidden_cat, pos_indices, n_matched, ntp_true, ntp_o1, ntp_o0, tok_ids):
    hidden = hidden_cat.to(device)
    W = model.lm_head.weight
    hmm_logits = (hidden @ W[tok_ids].T).float()
    lse = torch.full((hidden.shape[0],), float('-inf'), device=device)
    for i in range(0, W.shape[0], 2000):
        partial = (hidden @ W[i:i+2000].T).float()
        lse = torch.logaddexp(lse, torch.logsumexp(partial, dim=-1))
        del partial; torch.cuda.empty_cache()
    log_probs = hmm_logits - lse.unsqueeze(-1)
    probs_hmm = log_probs.exp().detach().cpu().numpy()
    del hidden, hmm_logits, lse, log_probs; torch.cuda.empty_cache()

    ntp_llm = probs_hmm[pos_indices[:n_matched]]
    n = min(n_matched, len(ntp_true))
    kl_llm = kl_divergence(ntp_true[:n], ntp_llm[:n])
    kl_o1  = kl_divergence(ntp_true[:n], ntp_o1[:n])
    kl_o0  = kl_divergence(ntp_true[:n], ntp_o0[:n])
    return kl_llm, kl_o1, kl_o0

# Warmup
_d = np.random.rand(3, 4, 4); _p = np.array([0.25, 0.25, 0.25, 0.25])
_tok = np.array([0, 1, 2, 0, 1], dtype=np.int64)
_ = full_bayesian_beliefs_numba(_tok[:3], _d, _p)
_ = reduced_resolution_beliefs_numba(_tok, 1, 2, 2, np.random.rand(9, 4), 3)
_ = reduced_beliefs_direct_numba(_tok, 1, 2, 2, _d, _p)
del _d, _p, _tok
print('Infrastructure loaded. Numba compiled.')


## HMM Definitions

In [ ]:
# ===== Spiral (2 tokens, 3 states) =====

def spiral_matrices(a):
    return [
        np.array([
            [0.2 * a,        0,     0      ],
            [0,              0,     0      ],
            [0.25 * (1 - a), 0,     0.5 * a],
        ]),
        np.array([
            [0.8 * a,        1 - a, 0      ],
            [0,              a,     1 - a  ],
            [0.75 * (1 - a), 0,     0.5 * a],
        ]),
    ]

def spiral_order_one(a):
    n1 = a * (35 + 23 * a)
    n2 = -50 - 55 * a + 23 * a**2
    n3 = 500 - 145 * a + 23 * a**2
    d1 = 10 * (5 + 9 * a)
    d2 = 10 * (-55 + 9 * a)
    return [
        np.array([[n1/d1, 0], [n2/d2, 0]]),
        np.array([[0, -n2/d1], [0, -n3/d2]]),
    ]

def spiral_order_zero(a):
    return [np.array([[(5 + 9*a) / 60]]), np.array([[(55 - 9*a) / 60]])]

# ===== Wing (2 tokens, 3 states) =====

def wing_matrices(x, y):
    b = (1 - x) / 2
    return [
        np.array([[0, b, 0], [0, y*x, 0.5*b], [b, 0, 0]]),
        np.array([[x, 0, b], [b, (1-y)*x, 0.5*b], [0, b, x]]),
    ]

def wing_order_one(x, y):
    p = 2 - 4*x + 2*x**2 + 3*x*y - 3*x**2*y + 4*x**2*y**2
    q = -3 + x + 2*x**2 - x*y - 3*x**2*y + 4*x**2*y**2
    r = 4 + 6*x + 2*x**2 - 5*x*y - 3*x**2*y + 4*x**2*y**2
    d1 = 5 - 5*x + 4*x*y
    d2 = -7 - 5*x + 4*x*y
    return [
        np.array([[p/d1, 0], [q/d2, 0]]),
        np.array([[0, -q/d1], [0, -r/d2]]),
    ]

def wing_order_zero(x, y):
    d1 = 5 - 5*x + 4*x*y
    d2 = 7 + 5*x - 4*x*y
    return [np.array([[d1/12]]), np.array([[d2/12]])]

# ===== Strata (2 tokens, 3 states) =====

def strata_matrices(a, t0, t1):
    b = (1 - a) / 2
    return [
        np.array([[t0*a, 0, 0], [0, t1*a, 0], [0, 0, 0]]),
        np.array([[(1-t0)*a, b, b], [b, (1-t1)*a, b], [b, b, a]]),
    ]

def strata_order_one(a, t0, t1):
    n1 = a*(t0**2 + t1**2)
    n2 = -t0 + a*t0**2 - t1 + a*t1**2
    n3 = 3 - 2*a*t0 + a**2*t0**2 - 2*a*t1 + a**2*t1**2
    d1 = t0 + t1
    d2 = -3 + a*t0 + a*t1
    return [
        np.array([[n1/d1, 0], [a*n2/d2, 0]]),
        np.array([[0, -n2/d1], [0, -n3/d2]]),
    ]

def strata_order_zero(a, t0, t1):
    p = a*(t0 + t1) / 3
    return [np.array([[p]]), np.array([[1 - p]])]

# ===== Arch (3 tokens, 4 states) =====

def arch_matrices(a):
    b = (1 - a) / 3
    return [
        np.array([
            [0.8*a, 0, 0, 0],
            [0, 0.2*a, 0, 0],
            [0, 0, 0.4*a, 0],
            [0, 0, 0, 0.6*a],
        ]),
        np.array([
            [0, 0, 0, 0],
            [0, 0.4*a, 0, 0.4*b],
            [0, 0, 0.3*a, 0],
            [0, 0, 0, 0.16*a],
        ]),
        np.array([
            [0.2*a, b, b, b],
            [b, 0.4*a, b, 0.6*b],
            [b, b, 0.3*a, b],
            [b, b, b, 0.24*a],
        ]),
    ]

def arch_order_one(a):
    d1 = 20 + 109*a
    d2 = -580 + 409*a
    return [
        np.array([[3*a/5, 0, 0], [6*a*(10+27*a)/(5*d1), 0, 0], [18*a*(-80+59*a)/(5*d2), 0, 0]]),
        np.array([[0, (10+101*a)/750, 0], [0, a*(560+1507*a)/(50*d1), 0], [0, (-1000-4690*a+3527*a**2)/(50*d2), 0]]),
        np.array([[0, 0, (740-551*a)/750], [0, 0, (1000+4290*a-3127*a**2)/(50*d1)], [0, 0, -(28000-39540*a+14147*a**2)/(50*d2)]]),
    ]

def arch_order_zero(a):
    p1 = a / 2
    p2 = (20 + 109*a) / 600
    return [np.array([[p1]]), np.array([[p2]]), np.array([[1 - p1 - p2]])]

# ===== Mess3 (3 tokens, 3 states) =====

def mess3_matrices(a, x):
    b = (1 - a) / 2
    y = 1 - 2 * x
    ay, bx, by, ax = a*y, b*x, b*y, a*x
    return [
        np.array([[ay, bx, bx], [ax, by, bx], [ax, bx, by]]),
        np.array([[by, ax, bx], [bx, ay, bx], [bx, ax, by]]),
        np.array([[by, bx, ax], [bx, by, ax], [bx, bx, ay]]),
    ]

def mess3_order_one(a, x):
    A = 0.5 * (1 - 2*a + 3*a**2 - x + 6*a*x - 9*a**2*x)
    B = 0.25 * (1 + 2*a - 3*a**2 + x - 6*a*x + 9*a**2*x)
    return [
        np.array([[A, 0, 0], [B, 0, 0], [B, 0, 0]]),
        np.array([[0, B, 0], [0, A, 0], [0, B, 0]]),
        np.array([[0, 0, B], [0, 0, B], [0, 0, A]]),
    ]

print('All HMM functions defined.')


## HMM Registry

In [ ]:
HMMS = {
    'Spiral': {
        'fn': spiral_matrices,
        'order_one_fn': lambda *p: spiral_order_one(*p),
        'order_zero_fn': lambda *p: spiral_order_zero(*p),
        'params': [(a,) for a in np.arange(0.01, 0.11, 0.01).round(2)],
        'label_fn': lambda p: f'a={p[0]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Wing': {
        'fn': wing_matrices,
        'order_one_fn': lambda *p: wing_order_one(*p),
        'order_zero_fn': lambda *p: wing_order_zero(*p),
        'params': [(x, 0.4) for x in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'x={p[0]}, y={p[1]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Strata': {
        'fn': strata_matrices,
        'order_one_fn': lambda *p: strata_order_one(*p),
        'order_zero_fn': lambda *p: strata_order_zero(*p),
        'params': [(a, 0.38, 0.54) for a in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'a={p[0]}, t0={p[1]}, t1={p[2]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Arch': {
        'fn': arch_matrices,
        'order_one_fn': lambda *p: arch_order_one(*p),
        'order_zero_fn': lambda *p: arch_order_zero(*p),
        'params': [(a,) for a in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'a={p[0]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
    'Mess3': {
        'fn': mess3_matrices,
        'order_one_fn': lambda a, x: mess3_order_one(a, x),
        'order_zero_fn': None,  # uniform
        'params': [
            (0.005, 0.01), (0.005, 0.02), (0.01, 0.02), (0.05, 0.02), (0.10, 0.02),
            (0.60, 0.02), (0.70, 0.02), (0.80, 0.02), (0.85, 0.02), (0.90, 0.02),
        ],
        'label_fn': lambda p: f'a={p[0]}, x={p[1]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
}

for name, cfg in HMMS.items():
    T = cfg['fn'](*cfg['params'][0])
    print(f'{name:>8}: {len(cfg["params"])} params, {len(T)} tokens, {T[0].shape[0]} states')


## KL + Extraction + R² (all HMMs, CSV-resumable)

In [ ]:
# ---- Resume from CSV ----
if os.path.exists(KL_CSV):
    kl_df_prev = pd.read_csv(KL_CSV)
    done_hmms = set(kl_df_prev['hmm'].unique())
    print(f'Resumed: {done_hmms} already done')
else:
    kl_df_prev = pd.DataFrame()
    done_hmms = set()

kl_chunks = []

for hmm_name, cfg in HMMS.items():
    if hmm_name in done_hmms:
        print(f'\n===== {hmm_name} (done, skipping) =====')
        continue

    token_names = cfg['token_names']
    n_tok = len(token_names)
    tok_ids = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in token_names]
    n_params = len(cfg['params'])

    print(f'\n===== {hmm_name} =====')
    pbar = tqdm(total=n_params * N_SEEDS, desc=f'{hmm_name} KL')

    for param in cfg['params']:
        label = cfg['label_fn'](param)

        T_real = cfg['fn'](*param)
        T_o1 = cfg['order_one_fn'](*param)
        T_stack = np.stack(T_real)
        pi_real = stationary_distribution(T_real)
        pi_o1 = stationary_distribution(T_o1)

        # Order-0 ntp baseline
        if cfg['order_zero_fn'] is not None:
            T_o0 = cfg['order_zero_fn'](*param)
            pi_o0 = stationary_distribution(T_o0)
            ntp_o0_row = next_token_probs(pi_o0.reshape(1, -1), T_o0)[0]
        else:
            ntp_o0_row = np.full(n_tok, 1.0 / n_tok)

        # Order-1 ntp lookup table
        T_o1_stack = np.stack(T_o1)
        ntp_o1_lut = np.zeros((n_tok, n_tok))
        for zp in range(n_tok):
            b = pi_o1 @ T_o1_stack[zp]; b /= b.sum()
            for zn in range(n_tok):
                ntp_o1_lut[zp, zn] = (b @ T_o1_stack[zn]).sum()

        for seed in range(N_SEEDS):
            tokens = sample_hmm_sequence(T_real, pi_real, SEQ_LEN, seed=seed)
            prompt = tokens_to_prompt(tokens, token_names)
            input_ids = tokenize_prompt(prompt)
            pos_indices, tok_at_pos = match_positions(input_ids, tok_ids)
            n_matched = min(len(tokens), len(pos_indices))
            pos_indices = pos_indices[:n_matched]

            _, hidden_cat, _ = chunked_forward_with_hooks(
                input_ids, [], PROBE_START, collect_hidden=True)

            beliefs = full_bayesian_beliefs_numba(tokens.astype(np.int64), T_stack, pi_real)
            ntp_true = next_token_probs(beliefs, T_real)
            ntp_o1_arr = ntp_o1_lut[tokens]
            ntp_o0_arr = np.broadcast_to(ntp_o0_row, ntp_true.shape).copy()

            kl_llm, kl_o1_v, kl_o0_v = compute_fullvocab_kl(
                hidden_cat, pos_indices, n_matched, ntp_true, ntp_o1_arr, ntp_o0_arr, tok_ids)
            del hidden_cat

            for vals, source in [(kl_llm, 'LLM'), (kl_o1_v, 'Order-1'), (kl_o0_v, 'Order-0')]:
                if len(vals) <= 100:
                    continue
                cs = np.cumsum(vals)
                rm = (cs[100:] - cs[:-100]) / 100
                kl_chunks.append(pd.DataFrame({
                    'position': np.arange(100, 100 + len(rm)),
                    'KL': rm, 'source': source,
                    'hmm': hmm_name, 'param': label, 'seed': seed,
                }))

            pbar.set_postfix(param=label, seed=seed)
            pbar.update(1)

        gc.collect(); torch.cuda.empty_cache()

    pbar.close()

    kl_new = pd.concat(kl_chunks, ignore_index=True) if kl_chunks else pd.DataFrame()
    pd.concat([kl_df_prev, kl_new], ignore_index=True).to_csv(KL_CSV, index=False)
    print(f'  Saved ({hmm_name} done)')

print(f'\nAll done.')


## Load results (after restart)

In [ ]:
kl_df = pd.read_csv(KL_CSV)
r2_df = pd.read_csv(R2_CSV)
print(f'kl_df: {len(kl_df)} rows, HMMs: {kl_df["hmm"].unique()}')
print(f'r2_df: {len(r2_df)} rows, HMMs: {r2_df["hmm"].unique()}')


## KL plots

In [ ]:
hmm_names = list(HMMS.keys())
n_hmms = len(hmm_names)
fig, axes = plt.subplots(1, n_hmms, figsize=(5 * n_hmms, 4), sharey=True)
if n_hmms == 1:
    axes = [axes]

for ax, hmm_name in zip(axes, hmm_names):
    sub = kl_df[(kl_df['hmm'] == hmm_name) & (kl_df['position'] % 50 == 0)]
    for source, color in [('LLM', '#1f77b4'), ('Order-1', '#ff7f0e'), ('Order-0', '#2ca02c')]:
        s = sub[sub['source'] == source]
        mean = s.groupby('position')['KL'].mean()
        ax.plot(mean.index, mean.values, lw=2, color=color, label=source, markersize=0)
    ax.set_xlabel('Position')
    ax.set_title(hmm_name, fontweight='bold')
    ax.legend(fontsize=8)

axes[0].set_ylabel('KL divergence (100-pt rolling)')
plt.tight_layout()
plt.show()


## R² vs k-suffix plots

In [ ]:
dist_labels = {'real': 'Real', 'order-1': 'Order-1', 'order-0': 'Order-0'}
dist_colors = {'real': '#1f77b4', 'order-1': '#ff7f0e', 'order-0': '#2ca02c'}
dist_styles = {'real': '-', 'order-1': '--', 'order-0': ':'}

for hmm_name in HMMS.keys():
    params = [cfg['label_fn'](p) for cfg in [HMMS[hmm_name]] for p in cfg['params']]
    n_params = len(params)

    sub = r2_df[(r2_df['hmm'] == hmm_name) & (r2_df['dist'] == 'real')]
    best_layer = sub.groupby('layer')['R2'].mean().idxmax()

    fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=True)
    axes = axes.flatten()

    for i, param_label in enumerate(params):
        ax = axes[i]
        for dist_name in ['real', 'order-1', 'order-0']:
            s = r2_df[(r2_df['hmm'] == hmm_name) & (r2_df['param'] == param_label) &
                      (r2_df['dist'] == dist_name) & (r2_df['layer'] == best_layer)]
            mean = s.groupby('k')['R2'].mean()
            ax.plot(mean.index, mean.values, dist_styles[dist_name],
                    color=dist_colors[dist_name], lw=2, label=dist_labels[dist_name], markersize=0)
        ax.set_title(param_label, fontsize=9)
        ax.set_xlabel('k')
        if i % 5 == 0:
            ax.set_ylabel('R²')
        if i == 0:
            ax.legend(fontsize=7)

    fig.suptitle(f'{hmm_name} — R² vs k (layer {best_layer})', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()
